> 在线讲义：[第 07 章 · 计算图与张量结构：一个 token 经过一层要算多少](https://wikieden.github.io/ai-infra-from-constraints/lessons/07-compute-graph.html) · 本书《从约束推导 AI Infra》

# 第 7 章 · 计算图与张量结构：一个 token 经过一层要算多少

配套讲义：仓库里 `ai-infra/notes/lessons/07-compute-graph.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 小例子 | $kn$、$2mkn$、$b_W kn$ | `matmul_cost(3, 4, 2)` | 8 个参数、48 FLOPs、16 字节 |
| 第三步 一层的参数 | $2d^2 + 2d \cdot 1024$、$3df$ | `layer_params(QWEN3_8B)` | 41,943,040 和 150,994,944，FFN 占 78% |
| 第四步 配对数 | 长方形 + 三角形 | `n_pair(1, 2, 3)`、`n_pair_bruteforce(1, 2, 3)` | 12 |
| 第五步 prefill 8192 | 三项相加 | `forward_flops(QWEN3_8B, 1, 0, 8192)` | 133.6 TFLOPs |
| 第五步 decode | 同上 | `forward_flops(QWEN3_8B, 1, 8192, 1)` | 20.0 GFLOPs |

Qwen3-8B 的配置与 133.6 TFLOPs / 20.0 GFLOPs 取自 AII 第 2 章 §2.1；m = 3、k = 4、n = 2 和 S = 2、P = 3 是编的小例子。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
QWEN3_8B = {"L": 36, "d": 4096, "f": 12288, "n_q": 32, "n_kv": 8, "d_h": 128, "vocab": 151936}

**函数 `matmul_cost`**：[m,k] × [k,n]：(参数数 kn, 运算量 2mkn, 权重字节 b_w·kn)。

In [ ]:
def matmul_cost(m: int, k: int, n: int, b_w: int = 2) -> tuple[int, int, int]:
    """[m,k] × [k,n]：(参数数 kn, 运算量 2mkn, 权重字节 b_w·kn)。"""
    return k * n, 2 * m * k * n, b_w * k * n

**函数 `layer_params`**：一层的 (注意力四个投影, SwiGLU 三个矩阵) 参数数。

In [ ]:
def layer_params(c: dict) -> tuple[int, int]:
    """一层的 (注意力四个投影, SwiGLU 三个矩阵) 参数数。"""
    kv_w = c["n_kv"] * c["d_h"]
    proj = 2 * c["d"] * c["d"] + 2 * c["d"] * kv_w  # W_q、W_o；W_k、W_v
    ffn = 3 * c["d"] * c["f"]  # gate、up、down
    return proj, ffn

**函数 `n_pair`**：有效查询—键配对数：长方形 PS + 三角形 P(P+1)/2，再乘请求数。

In [ ]:
def n_pair(B: int, S: int, P: int) -> int:
    """有效查询—键配对数：长方形 PS + 三角形 P(P+1)/2，再乘请求数。"""
    return B * (P * S + P * (P + 1) // 2)

**函数 `n_pair_bruteforce`**：逐个数：第 i 个新查询能看到 S + i 个 token（i 从 1 开始）。

In [ ]:
def n_pair_bruteforce(B: int, S: int, P: int) -> int:
    """逐个数：第 i 个新查询能看到 S + i 个 token（i 从 1 开始）。"""
    return B * sum(1 for i in range(1, P + 1) for j in range(1, S + P + 1) if j <= S + i)

**函数 `forward_flops`**：一次前向的主要矩阵运算：L × (投影与 FFN + 注意力) + 词表头。

In [ ]:
def forward_flops(c: dict, B: int, S: int, P: int) -> float:
    """一次前向的主要矩阵运算：L × (投影与 FFN + 注意力) + 词表头。"""
    m = B * P  # 本次参与投影的 token 数
    linear = 2 * m * sum(layer_params(c))  # 投影 + FFN：按行数
    attn = 4 * c["n_q"] * c["d_h"] * n_pair(B, S, P)  # 注意力：按配对数
    head = 2 * B * c["d"] * c["vocab"]  # 词表头：每条请求只算最后一个 token
    return c["L"] * (linear + attn) + head

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步：[3,4] × [4,2]（编的小例子，BF16）")
n_w, f, mw = matmul_cost(3, 4, 2)
print(f"  参数 {n_w} 个，运算 {f} FLOPs，权重 {mw} 字节")

In [ ]:
print("\n第三步：Qwen3-8B 一层的参数")
proj, ffn = layer_params(QWEN3_8B)
print(f"  注意力投影 {proj:,}，FFN {ffn:,}，FFN 占 {ffn / (proj + ffn):.0%}")
print(f"  一层 FFN 的 BF16 权重 = {ffn * 2 / 2**20:.0f} MiB")
print(f"  投影 + FFN 运算量 = {2 * (proj + ffn):,} × m")

In [ ]:
print("\n第四步：S = 2、P = 3 的配对数（编的）")
print(f"  公式 {n_pair(1, 2, 3)}，逐个数 {n_pair_bruteforce(1, 2, 3)}")

In [ ]:
print("\n第五步：一次前向")
pre = forward_flops(QWEN3_8B, 1, 0, 8192)
dec = forward_flops(QWEN3_8B, 1, 8192, 1)
print(f"  prefill 8192 个 token：{pre / 1e12:.1f} TFLOPs")
print(f"  已有 8192 个上下文再补 1 个：{dec / 1e9:.1f} GFLOPs")

## 3. 代码里才看得到的

1. `n_pair_bruteforce` 用两重循环逐个数"第 $i$ 个查询能看第 $j$ 个 token 吗"，和公式 `n_pair` 对上；规模大时只用公式。
2. 词表头只按 `B` 算（每条请求一行），不是按 `m`；如果误按 8192 行算，prefill 会多出约 10 TFLOPs。
3. 运算量只统计矩阵乘；归一化、RoPE、SiLU、残差这些逐元素运算没算进去（原文 L190）。

## 4. 自己改一改

- 把 prefill 长度从 8192 改成 4096、16384，看注意力那一项占比怎么变。
- 把 `QWEN3_8B` 的 `n_kv` 改成 32（不用 GQA），看一层投影参数变成多少。
- 用 `n_pair_bruteforce` 验证几组小的 $(B, S, P)$。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 节点算，边上传，箭头定先后；
> 行数乘权重，权重不随行数涨；
> 一层投影两亿个，FFN 占八成；
> 新旧握手长方形，新人之间三角形。